# Chapter 8: RAG Generation — The Answer Layer
## Topic 3: Faithfulness vs. Relevance vs. Correctness — Three Distinct Failure Modes

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- "The answer is wrong" is not a single failure mode — it conflates at least three genuinely different problems that require different diagnoses and different fixes. This topic exists to give each one a precise name, because different downstream checks and different specialists each target a different one of these three, and conflating them wastes debugging effort chasing the wrong layer of the pipeline.
- **Faithfulness:** does the generated answer accurately reflect what the retrieved context actually says, regardless of whether that context itself is correct or complete? A faithfulness failure is a generation-layer problem — the model said something the provided documents simply don't support.
- **Relevance:** does the generated answer actually address what the user asked, regardless of whether it's faithful to the context? A relevance failure can happen even with perfect faithfulness — the model can accurately summarize retrieved content that simply doesn't answer the question that was actually asked, which is really a retrieval-layer symptom surfacing at generation time.
- **Correctness:** is the answer actually true, according to real-world ground truth or an authoritative source, regardless of what the retrieved context said? A correctness failure can happen even with perfect faithfulness and relevance — if the retrieved context itself was wrong, outdated, or incomplete, a perfectly faithful summary of it is still factually wrong.
- The critical insight this topic exists to teach: these three can fail independently, in any combination, and a single "the answer was wrong" bug report gives no information about which one actually broke without a structured way to distinguish them.


### 2. Internal Working — Step by Step

**Distinguishing the three in practice:**

1. **Faithfulness check:** compare each claim in the generated answer against the specific text of the retrieved chunks that were actually sent in context. If the answer states a numeric value that contradicts what the retrieved chunk actually says, this is an unfaithful answer — a generation-layer failure, independent of whether the model's number or the chunk's number happens to be the real-world correct one.
2. **Relevance check:** compare the generated answer against the user's actual question, independent of the retrieved context's content. If a customer asks about one specific topic and the answer discusses a different, unrelated topic — faithfully summarizing a retrieved chunk that happened to be about that unrelated topic due to a retrieval miss — the answer is faithful to what was retrieved but irrelevant to what was actually asked.
3. **Correctness check:** compare the generated answer against ground truth — is the actual real-world fact what the answer claims, and does the underlying knowledge base correctly and currently reflect that? If the knowledge base itself contains an outdated document, a perfectly faithful, perfectly relevant answer can still be factually wrong.

**The four combinations that matter most in debugging:**

- Faithful and relevant and correct: the target state, everything working as intended.
- Faithful and relevant but incorrect: the retrieved context itself was wrong or stale — a knowledge base content problem, not a generation problem.
- Faithful but irrelevant: retrieval found the wrong content, and generation faithfully but uselessly summarized it — a retrieval-layer problem.
- Unfaithful, regardless of relevance or correctness: a genuine generation-layer hallucination — the specific target of dedicated hallucination detection.


### 3. How This Is Implemented in This Project

- **Faithfulness** is checked using the citation mechanism from the previous topic as its foundation: once an answer's claims are attributed to specific source chunks, faithfulness checking becomes "does this claim actually match what the cited chunk says" — an entailment-style comparison that can range from cheap and structural to more expensive and semantic.
- **Relevance** is checked independently of the retrieved context entirely — comparing the generated answer's content against the original query, which can reuse the same embedding approach used elsewhere in the pipeline to compute a semantic similarity between query and answer as a coarse relevance signal, or use a more nuanced model-based judgment for a deeper assessment.
- **Correctness** cannot be fully automated within the pipeline's own resources — it fundamentally requires either human-labeled ground truth or an external authoritative source to check against. This is the same evidence-based evaluation discipline used for retrieval quality, applied one layer further downstream to generated answers rather than just retrieved chunks.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Faithfulness and relevance are automatable at reasonable cost** and can be built directly into the production path, running synchronously on every request. Correctness fundamentally cannot be fully automated without an external ground truth, so it requires ongoing human review capacity, sampled rather than exhaustive.
- **Attribution in mixed-failure cases requires layer-by-layer discipline:** in practice, a single bad answer can have contributions from both the retrieval and generation layers simultaneously — a mediocre retrieval result combined with a generation layer that stretches to make the mediocre context seem more relevant than it actually is. Properly attributing blame in these cases requires checking each layer independently, not an either/or judgment based on surface impressions.
- **Monitoring:** faithfulness and relevance scores should be tracked as separate, distinct metrics over time, not collapsed into one aggregate — a dropping aggregate gives no signal about which layer degraded, while separate metrics point investigation toward the right team or pipeline stage immediately.
- **Debugging discipline:** when faithfulness and relevance both check out but an answer is still reported as wrong, the remaining possibility is a correctness failure rooted in the knowledge base itself, not the generation or retrieval logic — this should redirect investigation toward content freshness and provenance, not toward prompt tuning or retrieval ranking.
- **Cost:** faithfulness and relevance checks are comparatively cheap and can run on every request; correctness checking, requiring human review or authoritative external comparison, is inherently more expensive per case and should be applied through sampling rather than exhaustively.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **How much automated checking is worth building vs accepting some human review load:** faithfulness and relevance are automatable and should be built into the production path directly. Correctness fundamentally cannot be fully automated without an external ground truth, so a real, ongoing trade-off exists between accepting correctness risk as an ongoing human-review cost versus investing in better ground-truth infrastructure, like a more rigorously maintained, versioned knowledge base with clear provenance and freshness guarantees.
- **Where to draw the line between a relevance failure and a faithfulness failure when both layers contributed:** a single bad answer can have contributions from both simultaneously, and attribution in these mixed cases requires the disciplined layer-by-layer check described above, not a quick either/or judgment based on surface impressions.


### 6. Alternatives and When to Use Each

- **A single, undifferentiated "answer quality" score:** simpler to implement and communicate, but actively harmful for debugging — it gives no signal about which pipeline stage to fix, and different failure modes require entirely different remediation. Not recommended once a system moves past early prototyping.
- **Three separate, explicitly-named metrics (this topic's approach):** the correct choice once a system needs to be debugged and improved iteratively by different specialists — retrieval engineers fixing relevance-driving issues, prompt engineers fixing faithfulness, domain experts fixing correctness and knowledge-base staleness.
- **Formal, standardized RAG evaluation metric suites:** a more rigorous, standardized version of exactly this topic's distinctions, worth adopting once a project moves to systematic, ongoing RAG evaluation rather than ad hoc debugging.


### 7. Common Mistakes and Production Failures

- Treating every "wrong answer" bug report identically without first determining which of the three failure modes actually occurred, leading to wasted effort fixing the wrong pipeline stage.
- Assuming a faithful answer is automatically correct — faithfulness only guarantees consistency with the retrieved context, not truth, and a stale or wrong knowledge base can produce faithful-but-incorrect answers indefinitely without any faithfulness check ever catching it.
- Not distinguishing relevance failures from retrieval failures organizationally — a relevance metric dropping in production should route investigation toward the retrieval pipeline, not toward the generation prompt, but teams without clear metric separation often default to prompt-tuning as the first response regardless of actual root cause.
- Building expensive, real-time correctness checking that assumes it can be fully automated — correctness fundamentally requires ground truth external to the pipeline, and treating it as solvable purely through better prompting or better models is a category error.


### 8. Lead-Level Interview Questions

**Basic**

- Q: Define faithfulness, relevance, and correctness, and give an example where each could fail independently.
  A: Faithfulness asks whether the answer matches the retrieved context — a model stating a value that contradicts what the retrieved chunk actually says is unfaithful. Relevance asks whether the answer addresses the question asked — a faithful summary of a retrieved chunk about an unrelated topic, in response to a specific question, is irrelevant despite being faithful. Correctness asks whether the answer is actually true — a perfectly faithful, perfectly relevant answer can still be wrong if the underlying retrieved document itself is outdated or incorrect.

- Q: Why can't correctness be fully automated the way faithfulness and relevance can?
  A: Faithfulness and relevance can be checked using tools already inside the pipeline — comparing the answer to context, or to the query, using embeddings or entailment-style models. Correctness requires comparing the answer to ground truth external to the pipeline — the pipeline has no way to validate its own knowledge base's accuracy against itself, so correctness checking fundamentally requires either human review or an authoritative external source.

**Intermediate**

- Q: A customer complains that an answer was wrong. Your faithfulness and relevance checks both pass. Where do you look next?
  A: If faithfulness and relevance both check out, the remaining possibility is a correctness failure — the retrieved context itself was wrong or outdated. This points investigation toward the knowledge base's freshness and provenance rather than toward the generation prompt or retrieval ranking, since both of those layers performed correctly given what they had to work with.

- Q: How would you build a lightweight, production-viable faithfulness check without a full entailment model?
  A: Reuse citation infrastructure as the foundation — require the model to attribute each factual claim to a specific cited source, then perform an initial cheap structural check confirming the cited source actually exists in the context that was sent, followed by a coarser faithfulness signal such as lexical or embedding-similarity overlap between the claim and the cited passage. This won't catch subtle semantic contradictions as reliably as a dedicated entailment model, but it's cheap enough to run on every production request, reserving more expensive entailment-based checking for sampled or flagged cases.

**Advanced**

- Q: Design an escalation and routing policy that distinguishes these three failure modes automatically where possible, and routes appropriately when it can't.
  A: Faithfulness and relevance checks run synchronously on every request. A low faithfulness score triggers an automatic fallback — routing to human review or regenerating with a stricter grounding prompt — since this is a clear generation-layer signal the pipeline can act on directly. A low relevance score routes the request back through retrieval diagnostics rather than regenerating with the same context, since relevance failures are usually retrieval-layer symptoms. Correctness cannot be checked per-request in real time — instead, implement periodic sampled review by a domain expert, feeding confirmed correctness failures back into both the knowledge base and the evaluation set as new labeled examples.

- Q: A teammate proposes using a single aggregate score to represent overall RAG quality on a dashboard. What's your concern?
  A: An aggregate score is useful for tracking overall trend direction over time, but it collapses exactly the distinction this topic exists to preserve — a dropping aggregate score gives no signal about which of faithfulness, relevance, or correctness degraded, and therefore no signal about which team or pipeline stage should investigate. Keep the aggregate for high-level trend reporting, but insist the underlying dashboard break out the components separately for anyone actually debugging or improving the system — collapsing them for convenience defeats the purpose of measuring them separately in the first place.

**Scenario-based**

- Q: Over several weeks, faithfulness scores remain stable and high, but customer satisfaction is declining. Correctness spot-checks reveal the knowledge base's relevant documentation is significantly out of date following an undocumented policy change. Walk through the fix and the process gap that allowed this to go undetected.
  A: The immediate fix is updating the knowledge base with the current document and re-ingesting — a content problem, not a model or prompting problem, confirmed precisely because faithfulness stayed high throughout (the model was accurately reflecting what it was given, the whole time). The process gap: correctness checking relied on periodic, sampled human review rather than any systematic trigger for knowledge base staleness — there's no automated signal that fires when an underlying source document becomes outdated, only a downstream, lagging signal via customer dissatisfaction or spot-checks. A structural fix is establishing a knowledge base freshness and versioning discipline at the ingestion layer, so correctness risk from stale content is caught proactively rather than discovered reactively through declining customer satisfaction.

**Follow-up questions to expect:**

- "How would you decide how much of your traffic to sample for correctness review?"
- "What organizational process would you put in place so relevance failures automatically route to the right team?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **These three metrics form the conceptual foundation for more formal RAG evaluation frameworks:** several standardized evaluation metric suites for RAG systems are built directly on this same faithfulness/relevance distinction, extended further to separately assess retrieval-side precision and recall — recognizing this topic as the conceptual foundation for those more formal frameworks makes learning them later much faster.
- **The layered nature of RAG failure diagnosis is a general debugging principle:** breaking down a single vague symptom ("the answer was wrong") into which specific layer of a multi-stage pipeline actually failed is a broadly applicable debugging discipline, not unique to RAG — the same principle applies to diagnosing failures in any pipeline with multiple distinct processing stages.
- **Knowledge base staleness as a silent, structural risk:** correctness failures caused by outdated source content can persist indefinitely without any automated signal catching them, since faithfulness and relevance checks are both satisfied by design in this failure mode — this makes proactive content freshness management a genuinely distinct discipline from both retrieval quality and generation quality.

### 10. Quick Revision Summary (for last-minute interview prep)

> "The answer is wrong" conflates three genuinely different failure modes that need to be diagnosed separately. Faithfulness asks whether the answer matches the retrieved context, and is a generation-layer concern. Relevance asks whether the answer addresses the actual question, and is usually a retrieval-layer symptom surfacing at generation time. Correctness asks whether the answer is actually true, and can fail even when faithfulness and relevance both pass, if the underlying knowledge base itself is wrong or stale. Faithfulness and relevance are automatable and should run on every production request; correctness fundamentally requires external ground truth and can only be checked through sampled human review or authoritative comparison. When faithfulness and relevance both pass but an answer is still wrong, the investigation should point directly at knowledge base freshness and provenance, not at prompting or retrieval ranking — and these three metrics should always be tracked and reported separately, never collapsed into a single aggregate score, since doing so destroys the exact diagnostic signal this distinction exists to provide.


### Module 1: Setup — Four Scenarios Covering the Four Key Combinations

Build four concrete cases: the target state, and each of the three ways things can independently break.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Setup -- four scenarios covering the key combinations
# ------------------------------------------------------------------

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# ground truth: what is ACTUALLY true in the real world right now
GROUND_TRUTH = "The premature withdrawal penalty on Fixed Deposits is 1 percent."

SCENARIOS = [
    {
        "label": "Faithful + Relevant + Correct (target state)",
        "query": "what is the penalty for premature FD withdrawal",
        "retrieved_context": "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.",
        "generated_answer": "The penalty for premature withdrawal is 1 percent on the applicable interest rate.",
    },
    {
        "label": "Faithful + Relevant + INCORRECT (stale knowledge base)",
        "query": "what is the penalty for premature FD withdrawal",
        "retrieved_context": "Premature withdrawal of FD incurs a 2 percent penalty on the applicable interest rate.",  # KB itself is wrong/stale
        "generated_answer": "The penalty for premature withdrawal is 2 percent on the applicable interest rate.",  # faithfully repeats the wrong KB content
    },
    {
        "label": "Faithful + IRRELEVANT (retrieval found the wrong chunk)",
        "query": "what is the penalty for premature FD withdrawal",
        "retrieved_context": "Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures.",  # wrong chunk retrieved
        "generated_answer": "Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures.",  # faithful to what it got, but doesn't answer the question
    },
    {
        "label": "UNFAITHFUL (generation hallucinated, ignoring correct context)",
        "query": "what is the penalty for premature FD withdrawal",
        "retrieved_context": "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.",  # KB is correct
        "generated_answer": "The penalty for premature withdrawal is 5 percent on the applicable interest rate.",  # model made up a different number
    },
]

def normalize_vector(v: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(v)
    return v / norm if norm > 0 else v

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else 0.0

print("=" * 70)
print("FOUR SCENARIOS TO DIAGNOSE")
print("=" * 70)
for s in SCENARIOS:
    print(f"\n[{s['label']}]")
    print(f"  Query:     {s['query']}")
    print(f"  Retrieved: {s['retrieved_context']}")
    print(f"  Answer:    {s['generated_answer']}")

print("\nModule 1 complete. Run Module 2 next.")


FOUR SCENARIOS TO DIAGNOSE

[Faithful + Relevant + Correct (target state)]
  Query:     what is the penalty for premature FD withdrawal
  Retrieved: Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.
  Answer:    The penalty for premature withdrawal is 1 percent on the applicable interest rate.

[Faithful + Relevant + INCORRECT (stale knowledge base)]
  Query:     what is the penalty for premature FD withdrawal
  Retrieved: Premature withdrawal of FD incurs a 2 percent penalty on the applicable interest rate.
  Answer:    The penalty for premature withdrawal is 2 percent on the applicable interest rate.

[Faithful + IRRELEVANT (retrieval found the wrong chunk)]
  Query:     what is the penalty for premature FD withdrawal
  Retrieved: Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures.
  Answer:    Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures.

[UNFAITHFUL (generation hall

### Module 2: Faithfulness and Relevance Checks, Implemented

Faithfulness compares the answer to the retrieved context; relevance compares the answer to the query — these are DIFFERENT comparisons and can disagree.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Faithfulness and relevance checks
#
# Faithfulness: similarity(answer, retrieved_context) -- is the answer
#               consistent with what was actually retrieved?
# Relevance:    similarity(answer, query) -- does the answer address
#               what was actually asked?
# These are two DIFFERENT comparisons and can genuinely disagree.
# ------------------------------------------------------------------

def get_text_similarity(text_a: str, text_b: str) -> float:
    """A coarse semantic similarity signal using TF-IDF + SVD (offline
    stand-in for a real embedding model, same approach as Chapter 7)."""
    vectorizer = TfidfVectorizer()
    try:
        sparse = vectorizer.fit_transform([text_a, text_b])
    except ValueError:
        return 0.0
    n_components = min(2, sparse.shape[1] - 1) if sparse.shape[1] > 1 else 1
    if n_components < 1:
        return 0.0
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    dense = svd.fit_transform(sparse)
    return cosine_similarity(normalize_vector(dense[0]), normalize_vector(dense[1]))


FAITHFULNESS_THRESHOLD = 0.5
RELEVANCE_THRESHOLD = 0.3

print("=" * 70)
print("FAITHFULNESS vs RELEVANCE -- two INDEPENDENT checks")
print("=" * 70)

for s in SCENARIOS:
    faithfulness_score = get_text_similarity(s["generated_answer"], s["retrieved_context"])
    relevance_score = get_text_similarity(s["generated_answer"], s["query"])

    is_faithful = faithfulness_score >= FAITHFULNESS_THRESHOLD
    is_relevant = relevance_score >= RELEVANCE_THRESHOLD

    print(f"\n[{s['label']}]")
    print(f"  Faithfulness (answer vs retrieved context): {faithfulness_score:.3f} -> {'PASS' if is_faithful else 'FAIL'}")
    print(f"  Relevance    (answer vs query):              {relevance_score:.3f} -> {'PASS' if is_relevant else 'FAIL'}")

print("\nNOTE: this coarse similarity-based check CANNOT distinguish '1 percent'")
print("from '2 percent' or '5 percent' semantically -- both numbers appear in")
print("otherwise near-identical sentences, so lexical/embedding similarity")
print("alone often stays HIGH even when the specific fact is wrong. This is")
print("exactly why faithfulness checking needs claim-level number/entity")
print("matching, not just whole-sentence similarity -- Module 3 fixes this.")
print("\nModule 2 complete. Run Module 3 next.")


FAITHFULNESS vs RELEVANCE -- two INDEPENDENT checks

[Faithful + Relevant + Correct (target state)]
  Faithfulness (answer vs retrieved context): 0.648 -> PASS
  Relevance    (answer vs query):              0.511 -> PASS

[Faithful + Relevant + INCORRECT (stale knowledge base)]
  Faithfulness (answer vs retrieved context): 0.648 -> PASS
  Relevance    (answer vs query):              0.511 -> PASS

[Faithful + IRRELEVANT (retrieval found the wrong chunk)]
  Faithfulness (answer vs retrieved context): 1.000 -> PASS
  Relevance    (answer vs query):              -0.000 -> FAIL

[UNFAITHFUL (generation hallucinated, ignoring correct context)]
  Faithfulness (answer vs retrieved context): 0.648 -> PASS
  Relevance    (answer vs query):              0.511 -> PASS

NOTE: this coarse similarity-based check CANNOT distinguish '1 percent'
from '2 percent' or '5 percent' semantically -- both numbers appear in
otherwise near-identical sentences, so lexical/embedding similarity
alone often stays HI

### Module 3: Claim-Level Faithfulness — Catching the Number Mismatch

A more precise check that specifically compares numeric claims, catching exactly the failure mode Module 2's coarse similarity missed.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Claim-level faithfulness -- catching numeric mismatches
#
# Coarse similarity missed the wrong-number cases in Module 2. A real
# faithfulness check needs to compare the SPECIFIC CLAIMS (numbers,
# entities) in the answer against the SAME claims in the context --
# not just overall sentence-level similarity.
# ------------------------------------------------------------------

import re

def extract_numbers(text: str) -> set:
    """Extracts numeric values mentioned in text -- a simple proxy for
    'claims' in this coarse example. A real system would use a more
    robust claim-extraction step, but this catches the exact failure
    mode this scenario set is designed to expose."""
    return set(re.findall(r'\d+(?:\.\d+)?', text))


print("=" * 70)
print("CLAIM-LEVEL (NUMERIC) FAITHFULNESS CHECK")
print("=" * 70)

for s in SCENARIOS:
    context_numbers = extract_numbers(s["retrieved_context"])
    answer_numbers = extract_numbers(s["generated_answer"])
    ground_truth_numbers = extract_numbers(GROUND_TRUTH)

    unsupported_numbers = answer_numbers - context_numbers
    is_faithful_numerically = len(unsupported_numbers) == 0
    matches_ground_truth = answer_numbers == ground_truth_numbers

    print(f"\n[{s['label']}]")
    print(f"  Numbers in retrieved context: {context_numbers}")
    print(f"  Numbers in generated answer:  {answer_numbers}")
    print(f"  Faithful (answer numbers subset of context numbers): {is_faithful_numerically}")
    print(f"  Correct (answer numbers match REAL ground truth {ground_truth_numbers}): {matches_ground_truth}")
    if not is_faithful_numerically:
        print(f"  -> UNFAITHFUL: answer contains number(s) {unsupported_numbers} not in retrieved context")
    elif not matches_ground_truth:
        print(f"  -> Faithful to context, but context itself does not match ground truth (correctness failure)")

print("\nThis is the precise, structured version of the theory's four")
print("combinations, now backed by real numbers instead of just assertion:")
print("  - Scenario 1: faithful AND matches ground truth -- target state")
print("  - Scenario 2: faithful (matches context) but context is WRONG -- correctness failure")
print("  - Scenario 3: faithful, but doesn't address the query at all -- relevance failure")
print("  - Scenario 4: contains a number NOT in the retrieved context -- genuine hallucination")
print("\nModule 3 complete. All Chapter 8 Topic 3 modules done.")

print()
print("=" * 70)
print("CHAPTER 8 TOPIC 3 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  Faithfulness = answer vs RETRIEVED CONTEXT (generation-layer concern)
  Relevance    = answer vs QUERY (usually a retrieval-layer symptom)
  Correctness  = answer vs REAL-WORLD GROUND TRUTH (knowledge-base concern)

  These fail INDEPENDENTLY -- a single "wrong answer" report gives no
  signal about which one broke without checking each separately.

  Coarse whole-sentence similarity can MISS specific factual errors
  (like a wrong number) that a claim-level check catches directly --
  faithfulness checking needs claim-level granularity, not just
  overall semantic similarity.

  Faithfulness + Relevant + Incorrect = knowledge base problem, NOT
  a generation or retrieval bug -- this combination is only diagnosable
  by checking all three layers, never assumable from one alone.
""")


CLAIM-LEVEL (NUMERIC) FAITHFULNESS CHECK

[Faithful + Relevant + Correct (target state)]
  Numbers in retrieved context: {'1'}
  Numbers in generated answer:  {'1'}
  Faithful (answer numbers subset of context numbers): True
  Correct (answer numbers match REAL ground truth {'1'}): True

[Faithful + Relevant + INCORRECT (stale knowledge base)]
  Numbers in retrieved context: {'2'}
  Numbers in generated answer:  {'2'}
  Faithful (answer numbers subset of context numbers): True
  Correct (answer numbers match REAL ground truth {'1'}): False
  -> Faithful to context, but context itself does not match ground truth (correctness failure)

[Faithful + IRRELEVANT (retrieval found the wrong chunk)]
  Numbers in retrieved context: {'0.5'}
  Numbers in generated answer:  {'0.5'}
  Faithful (answer numbers subset of context numbers): True
  Correct (answer numbers match REAL ground truth {'1'}): False
  -> Faithful to context, but context itself does not match ground truth (correctness failure)

